In [2]:
%load_ext autoreload
%autoreload 2

In [3]:
from langchain_openai import ChatOpenAI
from langchain_google_vertexai import ChatVertexAI

from wsd.load_data import load_data
from wsd.models import BinaryWSD, ClusterByMeaningModel, DummyComparator
from linpub.metrics import accuracy

In [17]:
X, y = load_data(lang='fr')

k = 5000
X_test, y_test = X[:k], y[:k]

In [42]:
len(X), len(X_test)
X[0]

Candidate(pos='NOUN', text='objectifs', lemma='objectif', lemma_meaning='NA', example='Depuis combien de temps avez -vous revu les objectifs de votre programme de prestations et de services ?', context='Depuis combien de temps avez -vous revu les objectifs de votre programme de prestations et de services ?', document='NA')

In [36]:
from linalgo.annotate.models import Corpus, Document, Annotation, Target, Selector
from lineval.utils import Body
from datetime import datetime
from collections import defaultdict

In [37]:
Semcor_corpus = Corpus(name='Semcor')

grouped_X = defaultdict(list)
for row in X_test:
    grouped_X[(row.lemma, row.pos)].append(row)
items = grouped_X.items()
docs = []
for g, cands in items:
    contexts = "\n".join([anno.context for anno in cands])
    doc = Document(content=contexts,
                   corpus=Semcor_corpus
                   )
    doc_annos = []
    for c in cands:
        anno = Annotation(document=doc,
                          body=Body(text=c.text, context=c.context),
                          task="task",
                          annotator="none",
                          target={},
                          created=datetime.now())
        doc_annos.append(anno)
    doc.annotations = set(doc_annos)
    docs.append(doc)

Semcor_corpus.documents = docs

In [38]:
len(Semcor_corpus.documents)

1648

In [39]:
Semcor_corpus.documents[0].__dict__

{'id': '4421819a-b509-4fbd-856e-9d76782cd870',
 'uri': None,
 'content': "------Context------\nDepuis combien de temps avez -vous revu les objectifs de votre programme de prestations et de services ?\n-------------------\n------Context------\nAvez -vous fixé des objectifs spécifiques pour votre publication d' employés ?\n-------------------\n------Context------\nAtteindre ces objectifs ?\n-------------------\n------Context------\nCet objectif est respecté pendant toute la durée de l' action .\n-------------------\n------Context------\nComme première étape vers cet objectif , des arrangements ont été élaborés pour comparer les échelles actuellement utilisées par la circulation d' un groupe de thermomètres de résistance à platine standard pour l' étalonnage par chaque laboratoire national .\n-------------------\n------Context------\nSon objectif est simplement de déterminer < < quelles distinctions de longueur et de syllabicité il peut être souhaitable de rendre explicite dans une orthog

In [41]:
from wsd.models import get_data
get_data(Semcor_corpus.documents[0].content)

['Depuis combien de temps avez -vous revu les objectifs de votre programme de prestations et de services ?',
 "Avez -vous fixé des objectifs spécifiques pour votre publication d' employés ?",
 'Atteindre ces objectifs ?',
 "Cet objectif est respecté pendant toute la durée de l' action .",
 "Comme première étape vers cet objectif , des arrangements ont été élaborés pour comparer les échelles actuellement utilisées par la circulation d' un groupe de thermomètres de résistance à platine standard pour l' étalonnage par chaque laboratoire national .",
 'Son objectif est simplement de déterminer < < quelles distinctions de longueur et de syllabicité il peut être souhaitable de rendre explicite dans une orthographie de Kikuyu > > ( 59 ) .',
 'Il peut projeter des objectifs à long terme pour lui-même .']

In [23]:
list(Semcor_corpus.documents[0].annotations)[0].__dict__

{'id': '493c7ab7-2ac4-478d-9622-6f3c56705485',
 'entity': Entity::default,
 'score': None,
 'body': Body(text='objectifs', extras={'context': 'Il peut projeter des objectifs à long terme pour lui-même .'}),
 'task': Task::task,
 'annotator': Annotator::none,
 'document': Document::3f7fbd3c-4d59-4021-a8fc-4cdf449c8ffb,
 'target': <linalgo.annotate.models.Target at 0x7f3f1371f190>,
 'created': datetime.datetime(2025, 2, 5, 9, 22, 23, 267540)}

In [11]:
X[0]

Candidate(pos='NOUN', text='objectifs', lemma='objectif', lemma_meaning='NA', example='Depuis combien de temps avez -vous revu les objectifs de votre programme de prestations et de services ?', context='Depuis combien de temps avez -vous revu les objectifs de votre programme de prestations et de services ?', document='NA')

In [ ]:
# gpt = ChatOpenAI(temperature=0, model="gpt-4o-mini")\
# .with_structured_output(BinaryWSD)
# model1 = ClusterByMeaningModel(comparator=gpt)

# gemini = ChatVertexAI(temperature=0, model="gemini-1.5-flash")\
#     .with_structured_output(BinaryWSD)
# model2 = ClusterByMeaningModel(comparator=gemini)

# dummy = DummyComparator(probability=1)
# model3 = ClusterByMeaningModel(comparator=dummy)

# dummy = DummyComparator(probability=0)
# model4 = ClusterByMeaningModel(comparator=dummy)

# y_pred1 = model1.predict(X_test, verbose=True)
# y_pred2 = model2.predict(X_test, verbose=True)
# y_pred3 = model3.predict(X_test, verbose=True)
# y_pred4 = model4.predict(X_test, verbose=True)

# print(f"gpt-4o-mini:  {accuracy(y_pred1, y_test)}")
# print(f"gemini1.5-flash: {accuracy(y_pred2, y_test)}")
# print(f"dummy (always 1): {accuracy(y_pred3, y_test)}")
# print(f"dummy (always 0): {accuracy(y_pred4, y_test)}")

In [ ]:
# import pandas as pd

# records = []
# for yt, yp, x in zip(y_test, y_pred1, X_test):
#     record = {'lemma': x.lemma, 'pos': x.pos, 'y': yt, 'y_pred': yp, 'text': x.text, 'context': x.context}
#     records.append(record)
# pd.DataFrame(records).sort_values(['lemma', 'pos']).to_csv('plop.csv', index=False)